[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/1-1_%EA%B3%A0%EC%9E%A5%EA%B8%B0%EB%A1%9D%EA%B3%BC_%EA%B3%A0%EC%9E%A5%EB%A5%A0.ipynb)

# 1-1 · 하드디스크 6만 대의 고장 기록과 고장률

**⏱ 70분** · 모듈 1: 현장 설비군의 고장 데이터 → 경보 → 비용

## 🎯 이번 시간의 질문

> 데이터센터에 하드디스크가 6만 대 있습니다. **1년에 몇 대나 고장 날까요? 어떤 모델이 더 자주 고장 날까요?**

정비 전략(고장 나면 고치기 / 주기적으로 교체하기 / 상태를 보고 교체하기)을 고르려면, 먼저 **고장이 얼마나 자주, 어디서** 일어나는지 알아야 합니다.
이번 시간에는 실제 운영 기록으로 **연간 고장률(AFR)**을 직접 계산합니다.

**데이터:** 미국의 클라우드 저장 회사 Backblaze는 운영 중인 모든 디스크의 상태 기록(SMART)을 매일 공개합니다.
여기서는 2026년 1분기(90일) 동안의 **60,000대 표본**을 씁니다.

**이번 시간에 익힐 패턴:** P1 읽고 보기 · P2 골라내기 · P3 묶어 요약(`groupby`) · P4 막대그래프

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/backblaze_2026Q1.csv.gz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 1. 데이터 읽기

파일이 커서(523만 행) 읽는 데 **20초 정도** 걸립니다. 기다려 주세요.
(`dtype=`, `parse_dates=`는 메모리를 아끼고 날짜를 날짜 형식으로 읽게 하는 옵션입니다. 외울 필요 없습니다.)

In [ ]:
# ▶ 그대로 실행하세요
raw = pd.read_csv('data/backblaze_2026Q1.csv.gz',
                  dtype={'serial_number': 'category', 'model': 'category'},
                  parse_dates=['date'])
print('읽기 완료')

**P1 · 읽고 보기** — 표의 크기와 앞부분을 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(raw.shape)
raw.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 5,227,853행 × 11열입니다. **한 행 = 디스크 한 대의 하루 기록**입니다. 523만 대가 아니라 6만 대를 90일 동안 매일 기록한 것입니다.

| 열 | 뜻 |
|---|---|
| `date` | 기록 날짜 |
| `serial_number` | 디스크 고유 번호 (한 대를 구분) |
| `model`, `capacity_tb` | 모델명, 용량(TB) |
| `failure` | 그날 고장으로 교체되면 1, 아니면 0 |
| `smart_9_raw` | 누적 사용 시간(시간). -1은 "보고 안 됨" |
| `smart_5_raw`, `smart_187_raw`, `smart_197_raw`, `smart_198_raw` | 디스크가 스스로 보고하는 상태 지표 (다음 시간에 사용) |

## 2. 몇 대를, 며칠 동안, 몇 건의 고장을 보았나

`nunique()`는 **서로 다른 값의 개수**, `sum()`은 합계입니다. `failure`는 0/1이므로 합계가 곧 고장 건수입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
n_rows = len(raw)
n_drives = raw['serial_number'].nunique()
n_failures = raw['failure'].sum()
print('기록(디스크·일):', n_rows)
print('디스크 수:', n_drives)
print('고장 수:', n_failures)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 디스크 60,000대, 고장 200건입니다. 그러면 고장률은 200 / 60,000 = 0.33% 일까요?

아닙니다. **모든 디스크를 90일 내내 관찰한 것이 아니기 때문**입니다. 아래 그래프로 확인합니다.

In [ ]:
# ▶ 그대로 실행하세요
days_observed = raw.groupby('serial_number', observed=True).size()
plt.hist(days_observed, bins=30)
plt.xlabel('관찰된 날 수')
plt.ylabel('디스크 수')
plt.title('디스크마다 기록된 날 수')
plt.show()
print('90일을 다 채우지 못한 디스크:', (days_observed < 90).sum(), '대')

💬 **결과 해설** — 대부분 90일이지만 **5,407대**는 기간 중간에 새로 들어오거나, 고장으로 빠지거나, 다른 곳으로 옮겨졌습니다.
고장이 아닌 이유로 기록이 끝난 디스크도 있습니다. 그래서 "대수"가 아니라 **실제로 관찰한 양(디스크·일)**을 분모로 씁니다.

## 3. 연간 고장률 (AFR, Annualized Failure Rate)

$$\text{AFR}(\%) = \frac{\text{고장 수}}{\text{디스크·일}} \times 365 \times 100$$

"디스크 한 대를 1년(365일) 운영하면 고장 날 비율"로 환산한 값입니다. 서로 관찰 기간이 다른 집단도 공정하게 비교할 수 있습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
afr = n_failures / n_rows * 365 * 100
print('AFR:', round(afr, 2), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `AFR: 1.4 %` 가 나오면 맞습니다.

💬 **결과 해설** — **100대를 1년 운영하면 약 1.4대가 고장** 나는 수준입니다. 6만 대라면 1년에 약 840대, 하루 평균 2~3대입니다.
하루에도 여러 대가 고장 나지만, 한 대 한 대로 보면 고장은 **드문 사건**입니다. 이 "드물다"는 사실이 다음 시간 경보 설계의 가장 큰 어려움이 됩니다.

## 4. 모델별 고장률 — `groupby` 로 묶어 요약하기

같은 계산을 **모델마다** 반복합니다. `agg()` 안에 `새 열이름=('계산할 열', '계산 방법')` 을 쓰면 여러 요약을 한 번에 만듭니다.
- `failures=('failure', 'sum')` → 고장 수
- `drive_days=('failure', 'size')` → 행의 개수 = 디스크·일

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
by_model = raw.groupby('model')['failure'].agg(failures='sum', drive_days='size')
by_model['AFR'] = by_model['failures'] / by_model['drive_days'] * 365 * 100
by_model.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 모델마다 한 행씩, 고장 수·디스크·일·AFR이 생겼습니다. 모델은 66종인데, 몇 대 안 되는 모델도 많습니다.

**P2 · 골라내기** — 관찰량이 충분한(디스크·일 10만 이상) 모델만 남기고, AFR 순서로 정렬합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
big = by_model[by_model['drive_days'] >= 100000]
big = big.sort_values('AFR')
big
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**P4 · 막대그래프** — 가로 막대(`barh`)는 이름이 긴 항목을 비교할 때 편합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
plt.figure(figsize=(8, 5))
plt.barh(big.index.astype(str), big['AFR'])
plt.xlabel('연간 고장률 AFR (%)')
plt.title('모델별 연간 고장률 (디스크·일 10만 이상)')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 15개 모델(전체 기록의 89%, 고장 180건)만 봐도 AFR이 **0.29%에서 5.87%까지 약 20배** 차이 납니다.
- 같은 데이터센터에서도 **모델에 따라 정비 계획(예비품 수량, 점검 빈도)이 달라져야** 합니다.
- 5.87%인 `HGST HUH721212ALN604`는 1년에 17대 중 1대꼴로 고장 납니다. 이런 모델은 우선 감시 대상입니다.

### ⚠️ 분모가 작으면 비율이 크게 흔들린다

관찰량이 적은 모델까지 보면 어떻게 될까요?

In [ ]:
# ▶ 그대로 실행하세요
by_model[(by_model['drive_days'] < 20000) & (by_model['failures'] > 0)]

💬 `ST16000NM000J`는 고장 **1건**뿐인데 AFR이 **29%**로 계산됩니다. 디스크·일이 1,252(14대 × 90일 정도)밖에 안 되기 때문입니다.
비율을 볼 때는 **항상 분자(고장 수)와 분모(디스크·일)를 함께** 봐야 합니다. 고장 1건 차이로 결론이 뒤집히는 집단은 판단 근거로 약합니다.

---
## 5. 실습 과제 (25분)

### 📝 과제 1 · 용량별 고장률
`capacity_tb`(용량) 별로 고장 수, 디스크·일, AFR을 구하고, 막대그래프를 그리세요. 어느 용량이 가장 높고, 그 결과를 믿을 만한가요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 4절 코드에서 'model'을 'capacity_tb'로 바꿉니다.
# 그래프는 plt.bar(by_cap.index.astype(str), by_cap['AFR']) 처럼 세로 막대가 보기 편합니다.


✏️ **나의 답:**

### 📝 과제 2 · 사용 연수별 고장률 — "오래 쓴 디스크가 더 잘 고장 날까?"
`smart_9_raw`는 누적 사용 시간(시간)입니다. 이것을 연(年) 단위로 바꿔 사용 연수별 AFR을 구하세요.

1. `smart_9_raw`가 -1(보고 안 됨)인 행은 빼고 → `ok = raw[raw['smart_9_raw'] >= 0].copy()`
2. 사용 연수 열 만들기 → `ok['age_year'] = (ok['smart_9_raw'] / 24 / 365).astype(int)`
3. `age_year`로 묶어 AFR 계산, 막대그래프

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 1, 2번 코드를 그대로 쓰고, 3번은 4절 코드에서 raw→ok, 'model'→'age_year'로 바꿉니다.


✏️ **나의 답:** 사용 연수에 따라 고장률은 어떻게 변하나요? 어느 연수의 값은 믿기 어려운가요?

### 📝 과제 3 · 토론 — 예방정비(주기 교체)는 좋은 전략일까?
과제 2 결과를 보고 답하세요. "**사용 6년이 되면 모든 디스크를 교체한다**"는 규칙을 쓰면,
1) 교체하는 디스크 중 실제로 1년 안에 고장 날 디스크는 대략 몇 %일까요? (6년차 AFR을 이용)
2) 이 규칙으로 막을 수 없는 고장은 어떤 것인가요?

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 한 행의 의미 | 523만 행 = 6만 대 × 약 90일. 행 수 ≠ 설비 수 |
| AFR | 고장 수 ÷ **실제 관찰량(디스크·일)** × 365. 전체 1.4%/년 |
| 비교 | 모델별 AFR은 20배 차이. **분자·분모를 함께** 봐야 한다 |
| CBM으로 가는 질문 | 고장은 드물고, 나이만으로는 고장 날 디스크를 고를 수 없다 → **상태 신호**가 필요하다 |

다음: **1-2 · SMART 신호로 고장을 미리 알 수 있을까?**